[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/01_Python%E5%9F%BA%E7%A4%8E/06_pandas%E5%85%A5%E9%96%80.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 06. pandas入門 〜表データを自在に扱う〜

**pandas** は、Excelのような「表（テーブル）」をPythonで扱う道具。
CSVを読み込み、選び・絞り・集計する——統計・マーケのノートの中心スキルです。

**ゴール**：CSVを読み込んで、列の取り出し・行の絞り込み・グループ集計ができる。

In [ ]:
import pandas as pd
import numpy as np

## 1. Series と DataFrame

- **Series**：1列のデータ（ラベル付きの配列）
- **DataFrame**：複数列の表

In [ ]:
s = pd.Series([60, 75, 90], index=['数学', '英語', '国語'])
print(s)
print('英語の点:', s['英語'])

In [ ]:
df = pd.DataFrame({
    '名前': ['あおい', 'はると', 'ゆい'],
    '数学': [90, 60, 75],
    '英語': [70, 85, 80],
})
df

## 2. CSVを読み込む

実際のデータは `read_csv` で読みます。教材の生徒データを使います。

In [ ]:
df = pd.read_csv('../data/students_scores.csv')
df.head()       # 先頭5行を確認

### 📋 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

In [ ]:
print('行数・列数:', df.shape)
print('列名:', list(df.columns))
df.describe()    # 数値列の要約（平均・標準偏差・四分位など）

## 3. 列を取り出す

`df['列名']` で1列、`df[['列1','列2']]` で複数列。

In [ ]:
print(df['数学'].head())
print('数学の平均:', df['数学'].mean().round(1))
df[['数学', '英語']].head()

## 4. 行を絞り込む（条件フィルタ）

`df[条件]` で、条件がTrueの行だけ残します。NumPyの真偽値インデックスと同じ発想。

In [ ]:
# 数学が80点以上の生徒
good = df[df['数学'] >= 80]
print('80点以上:', len(good), '人')
good.head()

In [ ]:
# 複数条件は & (かつ) / | (または)。各条件は () で囲む
df[(df['数学'] >= 80) & (df['英語'] >= 80)].head()

## 5. 新しい列をつくる

In [ ]:
df['合計'] = df['数学'] + df['英語'] + df['国語']
df['合否'] = np.where(df['数学'] >= 60, '合格', '不合格')
df[['生徒ID', '数学', '合計', '合否']].head()

## 6. グループごとに集計（groupby）

**統計・マーケで最頻出**。「クラスごとの平均点」のような集計が1行で書けます。

In [ ]:
# クラスごとの数学の平均
print(df.groupby('クラス')['数学'].mean().round(1))

# 複数の集計をまとめて
df.groupby('クラス').agg(
    人数=('生徒ID', 'count'),
    数学平均=('数学', 'mean'),
    英語平均=('英語', 'mean'),
).round(1)

## 7. 数える・並べ替え・クロス集計

In [ ]:
print(df['合否'].value_counts())          # カテゴリの個数
print()
print(df.sort_values('合計', ascending=False)[['生徒ID','合計']].head(3))  # 上位3人
print()
print(pd.crosstab(df['クラス'], df['合否']))  # クラス×合否の表

> 📊 `pd.cut`（階級分け）も統計でよく使います → `02_統計_4級/01` で登場します。

---
## 🏆 練習問題

**問1.** `students_scores.csv` を読み込み、`英語` の平均と最高点を求めよう。

**問2.** `国語` が80点以上の生徒だけを絞り込み、何人いるか数えよう。

**問3.** クラスごとの `勉強時間` の平均を `groupby` で求めよう。

**問4.** `数学` の高い順に並べ替えて、上位5人の `生徒ID` と `数学` を表示しよう。

In [ ]:
# 準備
import pandas as pd
df = pd.read_csv('../data/students_scores.csv')


In [ ]:
# 問1〜4


<details>
<summary>▶ 解答例を見る（クリックで開く）</summary>

<pre style="background:#f6f8fa;padding:10px;border-radius:6px;overflow:auto"><code>print(df[&#x27;英語&#x27;].mean(), df[&#x27;英語&#x27;].max())
print(len(df[df[&#x27;国語&#x27;] &gt;= 80]))
print(df.groupby(&#x27;クラス&#x27;)[&#x27;勉強時間&#x27;].mean())
print(df.sort_values(&#x27;数学&#x27;, ascending=False)[[&#x27;生徒ID&#x27;,&#x27;数学&#x27;]].head())</code></pre>
</details>